# R06 — Controlled decoding-budget experiment

**192 GPU model calls** on 48 previously frozen external problems; T4 GPU; separate Google Drive checkpoint. Runs independently of R07. The conditions are source128, source256, source-conditioned finalization128, and regenerate128. This is an *equal per-call token-budget* and *extended-budget control*, not automatically an equal-compute comparison: report raw forward-token passes. The original R02/R03 studies remain unchanged.

In [ ]:
import os,sys,subprocess
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['TOKENIZERS_PARALLELISM']='false'
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.56.1','accelerate==1.10.1','huggingface-hub>=0.34,<1','pandas>=2','scipy>=1.12','safetensors>=0.6','PyYAML>=6'],check=True)
from google.colab import drive
drive.mount('/content/drive')
import torch
if not torch.cuda.is_available():raise RuntimeError('T4-class GPU required')


In [ ]:
import os,sys,json,time,random,hashlib,zlib,base64,zipfile,math,gc,re,subprocess
from pathlib import Path
from collections import Counter,defaultdict
import numpy as np
from scipy.stats import binomtest
import pandas as pd

def sha(obj):return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()
def load_blob(blob):return json.loads(zlib.decompress(base64.b64decode(blob)))
MANIFEST=load_blob('eNq9feuP5DZ2778iDMbwl562SL395WJsZ6/tXWed9eAmSBAElEh1ya2SKpKqa3oX+d9zHqReJc0oE+zFeu2WxOfh4Tm/8yDrb28K1Zv+zbfev/3tjVYDPAzw8OapP6fPbx68N5UeH78N/SJIQqMyP4n9spT4ve2qp6pR9X9UjTYfoWySwdtL154v1NDP7anxlG4vQ+8pT7dPj573o/EG9Wx6bzgZfOUNLf351LXt2XQP3u1UFSevaHuo9Fb4PtT5MH332rI0Xe+dqjM0Gfhfebrqi/baDF7Zdl5uquYJPjTm5hXXfsAq2Gl7885XaFa3tmdqDstiR/8HJzOo7onmn/iP/n89eJ8niV/qKDeFCpQOYl2KTZIIP40XRPlFdYWpve7aIFHyqngtauPBUDvz6P1Y9d5ZVY0H5fW1QLp1QLBTZ+DfrxcYfFtCHaDft94vH7578L775V8ePNVo70Nnnp9hQo9ErUtXFQbLto3Bgh60+zbyfaqADydVl0SIsaRquEVsTI3NUWdUO4xgKX5qqMVz2wynB89OpW9rrDK0g6qxpcD3eYyegoJP1A0Pvq76wWiYJvbe0gDO3s3AFBf99TwKEX3FH3HM9J4n15l+sLW5b+6MiuJcsa/H2ZpX2o3UqK7xSlgKrzd1zd0R/XtPX7uKxqoGnt6SKUQgZHSUMcrcVzJUsdGJzLI03GSMNBQLvvi/Ku8qGONNNbDssCvy6ytQtVAdcTZuBaArr87lhKtAr1PcIa7uSfEy+zzJW9sRTdsGiGOeTaN7auAE65l3LdCv857Ui6Hd9FZCNej2ZOoLvphREOhhXnnvIMmHqq5hhxm9JFEU+0cJlEbCxGGQlWFoosiY7Z2TLujzfa0qWOAzCQ/Y+p55gpmdTW0Gz7yY7hWG2TU4XRIEnVHPpeoHt/tVgzP4q2loC0FdYBicRw8zMrDmsOdColK/kgZHhUEYR1KVUgR5Alsl3pxSFC9lwW+nqm5PuL3C0HsF7oTBAT8PrVavMHDYbQm/fkDC03jzURxUZyQE11A9LWpjLidzs1PG99X8Pbe74uujSyaDUpd+WpgU5L80ent+/mJ6f6rbV01MCcIFl6tU3ZmFXAFS/hkXgwWd8WQkeVUuwJQwTGZUQxu1Z0FB/C65AK7k40quc3fIIMjxY2O4qMtJyyw8OGutTRiHaSITWWo/3BHxUi6X9b2Xq74qkOmq4trhllEdyvLBextEqOFA4sMuNvrucwi7+b1XmG5ALdCrGmZiVZ4EZQd/elXpvbZXD6bi5bCHH73frpdL2xvvj6ArGqBsCw8kQXRLJYiUt/aKfFL3LUhhoBB8Vlp7japq0DEDUUwBJWFzoFgoYSMZ5LuFOg6IoFhnRntuGffRBZYNRq3qeknv7CiTFUDsKCrjWJsgkIW/Se4gXXIZTBvYK2+7rr0ZDYN8iCOfmKWsXqyuIuJCqaoZDKmPTg2k9YT/FRAQxo5cCjS5EOd55j+voMtAfSGwcNKlQWIvWa6nmi/GVUXybGiP1A8OUkAERoS+ytNYK6WyaHubhQsCoE78ob3mQ4kCskcd2zFw0F31ghLxBHISxAewQuI9V8OAG49wGdCLNAWq0rotcM5NdVYkFusBN88JWAq+VjgvUhhQqCZSVQ3Q+Mxbk9QmFxtuoEPaK/BjoYaeqfM7QDH4S7vOmeS9eu255q9qKE6vDzQMmAQUhroPtCZYCwQeghT82lzPOaHAcfy2yQe7rT501cfKcFNuwLOmhHyc9IEjxYgKJyqW6lzVr14DRXEGy+UMDwvNxI/DrAhiAxLTFNviI06zFUBs2hp37vXpNDAP1y2glUt7YfVdtlecVaMrY+mH2xMY/1EiRCt5I7s6bu/6jyHsm9O4a1GZCH/VNLxwDVObgAO+TB2GflCIUCcmkHGqgm0IJES25OQ/IC/imvVFZRpY8qJWPeyy4aqBO77rlH71agRxRvPEaaY3ALMGBQ8BQqNosUMfoE0N4pMhc922mrmOReC8EQXcrrpnFgFL3Uqkosr4pLirR+97VRfXGkXIkifXPdp9AVuIeoCZ3KozmR2L0b6Yx6VSloeRlB9keZSGSuRFnqgy2zbLgjXqMLVaMBiA45pRZW/RJohQIi4+Adp6R1INhg26qq7PphlQT8mvJonq0ElxwmloqkgtXJtqmKsLLGeHgCKTynDrWCWwAnulsaPoKOMlSVGoMsJ/SV/ubLkVDvuLem2YbWIfYSRapnWNcAwWsAHF2vcVciBuEftApYNvwhWvMFSGp/eg2OtK0X58T38yOoe1ne9C0p1EAhKUFVk1gFRNUWnkT9bSQwtkwH3hJDF10mhiK8PaCqTVaG6tNi30eRjvxEUpwzTMtExEvA3MMxmu4A7tmndDCyPua0Qc5/YFiEJ6E17caku7tqlxwUEckV45kWV5veBuQJg6QCsAXv9ALHc2bE3C1gLbBuR7Ncw8B9xNiJqmI21EsI8sza66wPLdGvtkQBlcAGxiP2NbOJiirs45IDX+BHzczgfxvu+vtFMR10B12606z/e7bW8ET9g3KiK0J7gCCBrYG/c1Tqgem3FgiBtmAx9dIlcYZ1US+HdmYQGcaYrBk+HYGJAvZUqs1/3gskcyDbI0SKJcpX6cFzuejGQJvH4wTYPrh2hTeBcaLE7xOiByoP1PHEBg3Su6tup7ngUS6GRtOfYJoTk5rWN8V2fyAhXEVOgu8QA8A+Idat6MCm0JACkVrAaZgGA/o5m8HhgCZ3x/YidEiYYWOxNsAaswwUbwsVGsvlac1TDqR4J+1w7EXo8Dp+L91N1qRdKDK6IyE4lYxaWIRZDkO4aH8LMVEkTtZV0jNwB9jce4DEZbPA+I9S8A9l8JHN5AByKDh8A/KH+MqRkEsA6cF5GzIo/eP8/4G4RWp56AmKZCZULWA7WD3UIZ6m5Jg+CwaitkHMSBNACfs2RbFiVJsgRPaNuw9OwX26Yzl1oVhrY4SKfKuI2E7ELQWEPT6MGCGfXw0jImKwWf6+A7lHZVpx2V+T1RC7bhC9ALsC/oddqY1oZ9Bhg5bnlkv+GE9hoyKUnEt7FrjZgKwS0gzw5xOA81f2UIDSxdWYea7cFpBhhYSPYhfewMOhGx6Gx4ba3JOqEXbNbh67fRnYOUaeV6sjVoaNXd0EZH7Qoih4chjAi08PM0L4sy8LXZNnmyQK69qKBRq+alQhsgHM1WIAptvxcrgQCyKdzTOSIzFAzEphZ9oex/VR0Aw/e8S8gQd0R8AllPPk7oiLxkC/vYdiSX7ffOmNQkYlZ9w3igbm/qksVZQ02dAfMWqFfa60AcM3UGaqGHigFjkqmPyYRhp6v7Qr5OGnvV1f2G5SIPOz5kqtMkK7IozZJQltt2aLZUCO9r8xEFA/Q1IPd0BgE7Yhi0szQZ1i8gsFkB9Nbqhr2Aa0hOCrdbSQv0V/J9z4qaBjErouahnxQMUJrJzuwAygj268W0F7AKXmDxcDPU1D4R1TGNJe5khRLyIvFONgKXgxVmfqjGRefOBlw/NobBalqxxTBWZ/O4ban/dHR6Q6+z/gA25yho7ZaDrvoLmtD9zFjlSVNxovO9AzYQ2VHRCrI1CvxQZ8po5W+vrsyipbpXqG1H4er8S7DC5qPpiqq3tgNaSW1tyO/2iCKUZAqtybU/XS8Av8C46AGzDhaJSd/TgJByMkLAmOgZUk3eAHKH/tmZTEULdeAVOUWpcWlRu+2AUfu6hQcyxEpzQ/ty1rsGo7+2sHICausBsdr7movxruOwk5UWCjQJLDhju6/Rp9CMNim6RGAAK3QWHA40aT8rotRkWSZEKfNtIy9Klv7yHyrVGJYlys1r6NCrqFjOIbO9qPpKgAE9Bx9QN64+XerrLFhkm7kplDQvZKGAaoosaqDIzLWe1b6RWQRvLnXFZUcpOxUClA19oGOGgMVN9XbdGlTGboCz7mkWK3VzFFf5IpF54RdZ6ociSvIdpLuycH67/lU1jY02pT6sfPtcjaYyixWkAViPxfVSkJVC30L6xgKDIy2GPWTOSweKc7KzMfTCJke3HX0Z3Y21KQf23GGD6MyAemv+OqyBo8jP0BejQaRoX+yEY1ZI618V6Jr+QuJOfMMaGPlHgd2MwNtDaP4AnwJ8oDDW1dR2xzUK5fRpCgPOIIv1ulpzrFQvMJSB/QrV8EoS5alC8xI/a4VODKCB9P23WOdqgdXJev7R4PwIo+wtHEtsOax8q0rDkTaNqAyrsYHhsBs7EMmbecKlf2pbFP9gLtDske6EDqHJ0fM7MxV45Uhkj3EzWj5V4pBBd+DUYcCLERJ1EICulvNw+FEa6QemKHMRRYUJtpczWEUfr+jHIVTfMcZsrXzDxfndoHshN6ohY1N5v6uOFkL1z7wOTvGdRmCCyg2QcvPssDE1hPX+3Fjfb4rBjoaJTC+cHCf5PUodluCk5R345tLRV7Piq5IzIwXUqTNTnq4gzNe++YNkzSITlVJHmQiCON8hayiXvrbvSRMCrGFnDtqkmJvwO5GStaVW5OkE3jEYrhyhKMyMNFyBPqLm68FqXliGARgbJoyoqoN2uTFknd/bpyegJb0AoVM11mvOJShAToin5pZnXiYcGAETDimhrb1qn33MMrV+dmp21p39HoNo6vtpRZaNEM/0sBiLfIG3MW2pk/tEIqOcKVp4jeJtClUsOp4pn2Eqsux4sSnJvbdsw60jcDnNY2f8K845auDEcZH6RVL4URzqZNsXnoVLlfMzrDDazzUoHrsfMRRKcq1r2wF9I5d+il3F3qs6O7sO7czcqiuMKorHiLTMA0LVHrhn8C5A2aFdBhCkLYQUDVFkd+3QzyKA2JCMrEqDFevH+CzhcNztT2tPCcfXu5W+Do7aISEGAgOwC5NAJ9Ls6OtVQAyQ4i+AEUFGPHj/bHRjev4b5/WHruIHMv+fDAUX0Cb5Y3VuZyEA8Q4dah5HIIBMpKQ5LA/Nf7hym5x6c7p2tgcXSAQNL7mBjfo2iaY3Z4MggGJVHTtcRMw5CMj/PUZy663yo5Bl7ydBAxo+qWNrbVFyE3e+sgGTo27BQMQwbF9mZRGE2t+O+cfhKvhdV72ilByMtgG6Q2x9YTXKNnkI3EgjB+4dVFODouDiwIjAnxs15FRjtIe4GBq83COuxLw5UjkoPlYBrKM2kgxBeRayKLIy9ZMk2xH3SxvpT8DygLsB/z+pTqO1BGuDspvVfoNDA/VVAtBC9AdDrMC6bWmXjMYvevz79myN2vxa1aBYA5ogiGn8k1QmEOMfkKlmLQKPmHNvLe5OoZfYp+/E1Wg7o4+BMu/A/gTtDnaxHbIHergqzGQK23rYIrk4O1NyAGIcfU8uXG2cU8M5lWzXwipzbof9406sknZ7ZyNQGHWaLSyXRxHmhobwqBhYBNK6cuISEdjtKxs6YQ+9hZDPxmD482zlJ9aak3/lmj2Mr1QU+6WSSRDkWW5KfycpxF96rN6fXzmYDbLBvJoe/oYZX0AXD4iFyQXJkd22brt+MipYHqFH5HoBAjj/ObmMqFZffZy1OKuO6razpgaQ3Ki6J0lilavrGzAHjI3c+NDDHJTbtkipUt7fOIjZet11fW+qrCh9dAsGJg5VocIsyEoR78ifIL7zDD7bLDHOAKUs09wgcH+P/7Ww6y/tADgKhEhnw0rpzHcPSqEsSZupWZXiVJ2uCv6xNeR2DUrRm9XCCPs869WBUculs8oTthvBLLLLc+VCCjSLkfL3vTPtiQToHxpdaeFs7NgIDTJwQ8M3Kx1x1EmP+W6pjgOZGZOJItmJU6wTg9WZzD+QZb43ABw5cVYYKuIKvTVdhagiHN1lFK9Tl95LLOLHOtZMMLa4xfygOApoBgkMtamnr235Mc/D9VNWH1c5jIdlQOpnYWK0yOIIbIHteYdLj8wHEIu0zyiFszKcd3vDdJMQZdLg1a1FepjR1aHEOyuUvTWpBmtDseBjjE6V0WNSXAcOYVK7CAcxYoMpty3mBE8sw59RbeIObfPhzpdyGNYmgRE6BlBmTGy02lv69D5AYwV/BNZ9gVIFrUg2KzBlHHUAF5AL+46SgNCjhBOB/2JTs6nN2sLpuZau64zHoyksgc4AlIeJAjM6TPNt6Blkyb30sWm4LqydKzSCCzKnyQJivwG6fsZkx4EdEm337GyhVR4wQZoRq2Munu/PkCCB0Wnzu5y0nrN7e3ZFrHMqjmLwPPQjmfoyL/MkDba9xWm0REI/gXizqosTW19h+k+EIGAxQd/YBGSWl1RaTimws3Ksvh69XzAF7wzICnd2xYpsbmtOeTtT41waBQPIVEyaRGiFbhfUs/AFELtLpUyTxzFrFsby0zr9+6irtszLRKcAEUwoUn8n9TsTqwQKYP6uhlJPAOGrhsKnlJfDEyJcc0F5TqFBsrPUWeVtjioN0w2KEzxjMALhJLVgagy12AYwEdqR9GB9xOJSeGdDbsoLYUt0vDOVsEFmN7Ua+0qeHuUxlZSxiLTSmRAK9tweplrqke86Btu5QuOoYeexVucLMdUZ1P/rA8Lr0yhyFMj+0/Vc6aqskCIYGMZkFeAt4DMMOCFCdj79qrN5B9q8oPCZzENajv5bEK+3ByCSrq7nB5tb/wTKzPNoaB66dIy2OgebgMEa0NSU3jbmOJeUoTACb2gTIZYNoXEqjfBgaTgywUHF5TCdIOG8Rx7PXRvDzQlQss6pmVEA9euuH0bHCk7pSxpbDoPTLJgsnTUDFmvB2V8kyGzoZjEairch/2pdoXtU1avSy+7G4a8rRYtK87ktM3fY9TRhNJ7iuEiLoTNRZrm0uCSj9GfefMAXtIhrrHU4aqcyXwQgXZK0CON4D3PEG1CLThuN1riiVUJ6W2IoG6yzqodLO0scMRbKyy2vAxe29ZfTig/OKpF+UmaFjqPICLNrZy+91cBA+qxAx2PCYq7QeRB56oIR34sLzFgMbc4kIuor4qEWcBNZ1YiXOMReGQZOqcNGlFdnBlv/xbZI+82mDQDWxu8qr83YE3qWWV/3pnvhuC8YxS+cM8KtmWZMG0HhzFxaDmbRAtmIMKPGRqyHMQHbjg9auNgMZ0rRQbdRuPw4RVPmuQK2CNXgHmxAjnteceXR9ZNFEPhCiCDO06hI0x3v5nIBf1XD8Aqr8mt9PefkqaKsUcy6okNIL1VfobDmRHJnxSjMTIGt9cBhSTxL4dxEDx6sAyYbMVna8oFWpgYl54rj156jD7+3Ocbknz2J/kzmZ6Q2YfO3MSE3W3xxjIyGbce6Ipd/VNPlpQ6iOFaRymAf+9u5dsmSXB/aM4AaDLOAougUEMceczrRURoKjOFijqfauvbWjAHKACO+NYlrNKNOxoAqn8co7WdCntwVe3XDYGqJ4vQBFyQ2m9q5i3iRaOCGcLBrUklx9DhIpCIZCB1IVcSBSHeSm/0Va3XPmMR3Mje0IKd98XQ9s+SjJK4L5n3at6jWRTQripWRjvgROQwLL3aSKp7HNl18zxnctbKJgWDb3kf9j1pXYVgGUSTLoghCFWTxzhnCpeP7V0Rr1gM9N5b6FscLTz/jrAl1UHBtTF0mE6LqT/z4O02XytokOZzZosWpMLddzfx6xEe/L2vDnuIqD5gzM0ueZzU75V3YoWIclLugQKK1ZzjXbZ1SedRmN5HOyyLKQ4DmZZ5ukzRbxRJ+xmgshyN7T34TjBQ4G9XzmdH+QrlX+GF8V1wvvbPkXETA5rRe7VmLp6oEcf+r0V0L5vpdc25751U3nAjafZpwVzzKUsGuXYx4UgRfFn8RkcnCIohlkpWRKbftmWh1ggWm/QuleHCu9/cgKbqq7V06sMUe4xFNDJt8DXuoxqRIijCxY+sbBiVMRGpJt811+Gw7T5jkIh6TVTucXTKeoQQIV+HBOjpnYrHajbNY1UJdRzYkM/fN4XKO/XrtZSNeLw6bPrEOlB8apWQCWnQvGSvYyPS1UfdvyRXy4P1cWSPkQ9sszhBcakTllNEIFh8hjsDr2HtIDhlibszhBijUU0NQgJ9mRjb3MnYgnb09+mjYI/OT9V+CWh1O3M2DXTMcCKdg85nQj0OnuJ/x6ANtdcrflL4dwsLlSWPSrc2eQCpwo/aomYv6zIDVrBeGVzkuI4iXoaty2DarNLr0qK2fa51maWrSAuS0v7M3kiBdh8cKF7Hl8AjgEEvf3waEhY/e97Xq1Cg2llKai8OTKwzbYyqP4R4sMPMMzTohvewKv9g1owGtsqnuCdC/gFU9EYAevw1kkATGT9Pc90sV7h3yWjqGvmvznCMhUs5VLrAVJ/eggMPw1/1HmzJLJSKa2Jg4YHNr+gnGu2qcK7UGweNXHAiLlylZnhLUVkdYj5IkMCZOpFBlKopAB9sh0yW6+4lS8ijaOwsFg8yFZbMhZhklnNP76IW+TQfmHN/fga2th8EFjH9cLD0Xo6Wm5kYnuO1xI0k4Sw/O1sRZkCRGpMoYmaudxLFoFQhjIquuwNjhD+ShVHhAGU+7D3gsfOBEEULjxAn27ZioAvO4X3k5FluuuHuLAJ56syaFa2nlsTo4ddjvkZKmzIsyTFUhj8B4Ctc64KlfdG99uUnszps92qMoHbk41XS5CaZoRuOpNHfeiNqa5e6TN3N+HtlOcyy8mqs4OFmtQ11kReSnWW7EjitcroI8ocCgU60xaZhszq7S9jYKtqL7MdkpxyMTQ3vh1R5rPbUDnXgfi3+wp5FHuzedCtOiz9teboPlUKZyoAxK1l+zYbg4td2KTXv7fHhsk3BZGvggb2RcqCD2/W23phRidQoWAwaX67ApBsfTfuS1ABmhnmZIU2MKybrGrvCzmXnEHtDOZy2VbebwkyRUMouiMlTpzlGBJNqY4o0AqtKc9k75CcucT1RbHOjCv1hpoL17Iac0H/7FCHjoA9ZDfz4QYmDLvqQe4jhkDxeQqgQc3J94oandkSxjiygiKJNMaefMXOUtBfIgVcrM13EZFcIHs7AU2Z5De0M3onoTyafVn1jqP5jaUBFXII2C/wELLG912YnYHwUEIjFKxn6WaAAFRbBzjjC+u73gD3iQkQ6Fmtr6XoArekxRsOG0kM8/0HzleLoDk0Q6zImk825U26lwdeboyWzmUx3iHGzQw7TeL1MAoa/zLAzKNBd+GIq9iMUStr/XyrobfHZX9mxnVxaCjy+Xwovf8lrZJhx849pftnODsChTbUopIh2LfCd4Hy+37o/26MvNNGNKDl22ABsOPXh8BddJdS/GXvbkAhyce+QO2tgSqNZ65065Nl11mZpC5DP7uvhmt+f8gMxmOTL4JsOCrYLERQ5KN4zVNo+PckGUmERqzFCJ0C+/jfekn6588tbkEsKeR7kgR5qVGfATpi+4kokrMzeGbC3iC661BeeOsgPYM1kiMt8kAGzyHewqVgfx3uOlKvqKthSeCYQBAR6lTUYu28DrTxVebkP58RWY74ZkNOWNxe4jXxqzSO7Dkv14GMS1a2+0Wk3wKGSL8yCTidIhovSdtRKrOPav6loTEMELEz6yhwoxSqdet7w0j94P7iKxEWFjVNtzB0AiOdZ1Jxxs6jwnQFDYC7ZFFESuoBPydFYU5XuYTY0ssa57O1q51Baea6iQOdRtrdCiNDkqLPKykEks0jLOQhnIHe5YCfeqLHFclKJiz9uid8xlv3cGr1+jOMWfG/Qeo1Ont0gmIGxjVZW9+60BY95zhVisJO505FSWM5jD8V3eDsNSrC5Ku2wYlhD1jp/RP0ioJBJhLlQAqlAlOtoBy2LluVVFy4o9G08XsZ8W6BCNr3C158DFvra4ZXSAIMQx0GFzt1eio6ocIFCR+EKlqUj8PCx3sl+W12iY3BSFmlJX+ktNYZzZUWt3+Rqh14DuOEBPqavKuv6sury2FACkgxXmCYfj5W2kx6mJDbl3VITnMizixAeoGIsoV/GOWFgitZhc3dZTLW3OKpkWfcWBcnf8EP2Lj+MRXpvbeiYLDW13ChSu9b1tm412qqJ2Wl7x6GGtFZQG/udLUURmB7KvHFe/LUyzowadXFlfZNI1+xbd3GicduK2XfeZwWxZdis1f9Srk4MFJ/EodJlmUSZ2Mn6DFdYDW73BAJrNdZuOb+OVMPQRmYdUvCs1nomJpzPd9jYQgsIz+TU78W2bsGnOrvCXsUYp4ygRZZiKMs/Lvan64epocEPSiHZ5b8UtJ2zSse4OACwlrI9YMI7Ht5yLmi1rDeM0Hr1/xEu4LAyK7xqP06kl68afxM2MW8ZCSDFNA2Z5P3Cm7ZdhijyTqfRNWEoTpCbbTlGWYfL3hNCY/BBP8Ne2coeM7d0AE/J26TqT5901x4nLIzOKcCOQedgmBCQThFFQpGHsF/G2ISxXQdxf8AwpJc8884W+GGd2t78WiO/55D0OFEA0xdQQml2fAGohV2Tju7Jur527+otvJLhQsizmwywKjTegzVu78yMtuhpT3V3wV2n9ZfuuCNJChCILojAIRbGTFrCkUuR1dI8BWwE0egHqEHHf+HI6e5mr/nmRuM/u4UVx0jiLRtf1P5vks+Mt1VmqytwIMJTSvQtfs/igVJntiMDfExwToMYw0Y3ArxcHlOHJR05mgiWUi2Y+JUhsQGUqPAkUdGLZjqwLEUOaI1KdDW7tVjoKwovQJL6K/CBTgdiJFa6kDcXxYdeYClPUAr4OY4zY20tzHwB1LT+QpdzgvSV8c8Xyq81fmqMy/s76aOpx1Q8nNXHl1UY5ulPSQIRFKYvcBCoo4+2oU7zyRQd/d1+0FEd80XvtOd7YBiryqDpKkkBEmZYmDQVY8du2R3jvgPuL+mvb0VVowzuyyUGoAQnISn/rRWKMPCwuzw7G0noyet93z2BTKXs00nxEC94gTCfQC3pmeYGUu/GIz4dxc59P2N+cvFJFFvppnIgoSaIs3NE0S+gSOoCNUMIB708AeblA6J9C8s5AOITcj/poAGskSZrrPFRRLnascJlG/0t4tpaqqFrCGWLbgWfJXa00PobO7oQqyc8NzFZWnxKk4qhCSpOwKExagBzN0tLfQ/QrSWq6zoau6b4Xvk/q2pnx8lSOs7cU0D3hkUg7Z/y1gUfvT5RCjLJVa7q1ip27y5Ysgqfq9wQacwVWtVxUi67gwKpfBtZMUposzaXIs1KIdBvOhkvu4nwSowZGaidDN8zR9Y18w9EK5X53LRDUndQZTyCqgnEdXnbwF8Qc9fJy1kAknA2KJwvddttshsuTD24MorJkzqKpCf7NA/NC17Qh+l1SeCxHf9jA+n1XtJG5t8VlQ/OWv1CE55HOjFCRiEM/EfF20qOUdwYYnzqClbb3Z07XK4Ccng5A2Bt+0YK097PiQRqUwqPXcnaCFqpTSufddTRVv3Ww86j7IctFkfnSFzlYB+n25YYyWp0ZvOIdqpTrgwnSTxz5Awz/XGnacuw9ZKR/Vywciw18dN9eQX1B5pz/dsXsSlHKmsIm2obvUUcjaHF9tJ58h1SY+tpzIMqjsilUKtV4WURZ6ESJbVMg9Dc2IQpHSiFE6LIHi53t23vBZ0HvFFaW0afL4qG2Gs9CTC6KFUTmO50+IbyP6vlIpCpVflzKtIhKoXcib+F2zCXZD7lYy5GvEpt//rnC6MQYYSE/xGfCMba/tVPyqMcpyXWkghR0fSr9Ii133DAHQjGE4TIXaXmlKyTc7yuEoXtvf3Lip2EjdKPGwM1dfMbdNziPz0x9fPbez53IeSAFKJ8kLZMg2wmsLhWzjP/n8D5IP+OavMfsapk5cpcWEh9OnAuQfY0ulUxFGu/9/om/xumLpE66kcrupOHV2llVN7vV8sMY9iSBQHcH4PkTPLjSP4zXTjvBjzaSqhZJSDjjmXTMbIn1BZrzWwrnfWBK0hh7/TJukCaI8GbQspB+ptS2RRMFq+wSewto6f0RYAkLMMygo7NQfG82v4wDT/f8/s4xQoX5bhXbiIu5z256/DJ3SJRqmaV+UBq/9FO1EzXOsp2ocbASYXyYf54psIgjz+RWtqiT3qcbzDMNdkTYYUM9BwyTmziNwzzGs+I7XC7+P6LJxI/+t2hSZOITaJKkapQsS+CHMRihhr8T5JRhtOGmjY5q1bQ0sVYY4yjyQuttyzIWK9j5goYZ/D8SdPnmuytjg6fXs/3tBqtYppJhNt3V+W68rPNfFR5jmqdzjN+xkv1M5LVNtbtAKzoqW6IyEWWUpaoAyB3s/NyMTNO1GLaCEa8ttUcIBmuaWbn4NQbi/4pn1zEly6YBfG3vbCaGosPoHZ6NH/gQMkoodziQ25oBKXwk8UPVrBcb//35IwfbnpLQFBIPwkipQ7ODMKV/H6G2adt8hdg8Tp1gnk49jCHpMZNbRBulo/Cu9KhAFoX50B11vJmYlhz2rmdSgTXl5wAsgmwnMU2umdtmVCYoFntnXtPPZLhjFHiw2AFul4CZrsP0VNtC4C8OEARaZEmUB0km/UgmO3ahn6wWrXriC0Jmd4vSjdN0pwjGPW4uBDf/kYlX/o5OighFrSbWzmx5vobmJxs2okTV5e8ijDX4alOqRDfizHCC0bOQFI9mWSE3X5iZF8RJrAsgkxBBlukd7BytRJk9fHU9469M0tkeLxWZH6Vs55Hj80/qxueFKCn7FSAj3ZPpLhyYfnMojlNYIq7ZD+qVfoFpjTD464UvxKZLn+edu4v7sfECRqk6hn0cU7idWmoY3mHTd1dZJXQq7t/hraXWf6CaoHvtocDf3nwPmLY2P3zz2/97/8uvFEsyfl4KE5mk8E1WhDlsGUCgWsqwDAXgrxSs9cQwMWHMqvqGD8VA3ST0AyGBs1OA7EA0PNcaR2nqxwWY+gVg3CjP0jh+A8v3BqShOSus9k/i3T/8y4d3hAt6Y+1I8smmaGBU5+v53W8/vpdRTPcoeP+Jx5TR1Kzo/h48ee/yAJkh/uu/AYJAYjQ=');CASES=load_blob('eNq1W1mP5EZy/ivEYAb70ltLJm+9GKOVZUlrWYudAdawYRjJzGRXqlkkzaNrag3/940jySLZLIsewBAgTCXzivuLiOx//+93Wg6yN8O7b7x3z/0le3n35L2zev75TeSrMI2MzP008ctS4Pems8+2ltV/2lqbLzA3zWG07ZpLSxv91JxrT+qmHXpPerp5PnneD8Yb5IvpveFscMgbGvrnc9c0F9M9edezVWdPNT0seh/4Pqz5fP/uNWVput472wtsGfofPG171Yz14JVN5xXG1s/woTZXT439gEvw0ObqXUbYVjfuZNoO5+JB/4DEDLJ7JvpT/+T/z5P32yzxSx0XRslQ6jDRZbDLksDPkhVTfpadMpXXjTUypbDqpirjwVU7c/J+sL13kbb2YL4eFfKtA4adOwP/v7Vw+aaENcC/b7yfP3/75H37878+ebLW3ufOvLwAQSfiVttZZXBuUxuc6MG+72PfpwX44yyrkhgxz5Q174ibyXk7OoxWRzGI4seadrw09XB+8hwpfVPhkqEZZIU7hb7Pd/QkTHymY/jyle0Ho4FMPL2hC1y8qwESV+f1fIsg/sAf8c40zsR1ph/caj6bD6OpSCuedVrI3OrppkZ2tVeCKLzeVBUfR/zvPT12lu4qByZvrRRBGIj4qGKUhS9FJBOjU5HnWbSrGFkUrPTin2TRWbjjVdYgdrCKYrwBV5XsSLPRFICvLJ32jFKg4QwtZFp7lixmn4m8Nh3xtKmBOebF1LqnDc4gz6JrgH+d9yxfDVnTewHL4NizqVocWHAQ+GFubDvI8sFWFViY0WsWxYl/lEFZHJgkCvMyikwcG7NvOdmKP3+spAUBX8h5gOl75hkou5jKDJ55Nd0NrtnVSC45gs7Il1L2w2T9skYK/mZqMiFYCwqDdPRAkQGZg81FxKV+4w2OOoMoiYUsRRAWKZhKsktSnKx9waezrZozmlcUeTfQTrgc6PPQaHmDi4O1pTz8hIyn+xazO7AXZASvkD0JtTbt2VwdyThul+O870avj4pMhKUu/UyZDPy/MHqfPn9F3j9XzU2TUoJzQXGVsruwk1Pg5V9QGOzojCdiwVJpQSnhmqyohgy1Z0dB+i54AkrytPHrfBwqCGr8vBkKdU20yKODVGttoiTKUpGKUvvRAxcvxFqsH71C9lah0lk1dmgyskNfPnjvwxgjHHh8sGKj33yOwJo/esp0A0aBXlZAiQt5AoId/NOzpXdrRg9I8Qqw4ZP3aWzbpjfenyBW1MDZBn6QB9ENzSBWXpsR9aTqG/DCwCH4LLX2amkriDEDcUwCJ8E40C2UYEgG9W4VjkNiKK5Z8J53RjtqQWxwa1lVa37nR5VMAbPjuEwSbcJQKH+X3WG21jIgG9SraLquuRoNl3xKYp+UpbSvLlYRc2GWrQdD4aOTA0W9wP8ADIS7o5YCT1rSPM/81wixDMIXAovJu9TI7LXK9bTy1UxLkT070SPzw4McCEITRL4sskRLKfN438yiFQMwJn7XjMVQooPsMcZ2DBx0Z1/RI57BT4L7AFVIvRc7DGh4hMuAXxQpMJRWjUKaa3uR5BarAY3nDCoFXy3SRQEDJlXEKlsDjy9smhQ2edpwhRjSjKCPSg49c+dXgGLwLz0dzizv5a3nlX+WgzrfnugaQARMhrVPJBNcBQ4PQQp+rcdLQShwvr/b8smZ1efOfrGGt5ouvNgqEKd7PJhYMaPCOxdLebHVzathKlKwFmd02GmmfhLlKkwMeEyj9t1HkuUbgFg3FVru+HweWIerBtBK27QcvstmRKpqbY3jH5onKP5JIEQr2ZCnNZPt+qcI7OY8Wy0Gk8DfbA0D08a0J+CArwuHkR+qINKpCUWSyXAfAgVBvtbk71EXUWa9sqYGkatK9mBlw6hBO77tpL55FYI4o5lwovQKYNag4yFAaCQJO/IB2lTgPhkyV02jWevYBS43kaDtsnthF7COrcQqWoy/JB918v4oKzVW6ELWOrk90dkFmBCdAJRc7YXSjtVtX81pHZTFYSTlh3kRZ5EMClWkssz307JwizpMJVcKBuC4YlTZO7QJLpSYi78Abf2evBpcG2JVVV1MPWCcEh/uHnVCJ+qMZGhaSDuMtR2W4QLnuSugy6Q5vDsuCZ3D3kTsOD6qeGmqlCxj/J/wxQOT2+Cwv8hbzWqT+AgjMTOtKoRjIMAaAmvfW9RANBH3g2aHf4g2usJQGX59hMBeWUn2+JH+yegcZLu0QoqdxAJylJayGkCqRlmN+slRemiADWgXkyemQ2pNamU4WoG3mtOtjdHCmYfxTqJKEWVRrkUaJPvAPBfRBu6Q1fx+aODGfYWI49K8AlMobsLAtXK8a+oKBQ7uiOLKmTLLsUVrQJg6wC4AXr8nlbsYzibBtCC3Af9uh0XlgI+JMNJ0FI0I9lGm2dkWxHet3S8DwaAFsInnzHvhZVRlLwUgNf4EetwsL/Gx70eyVMQ1sNwdKy9Le3f7zeAJz8ZAhPkELwBHA7bxdsUZw2M9Xwxxw+Lic0lkhHvaksD/lBYq0EyjBk9E82bAvow5sZX7QbHHIgvzLEzjQmZ+UqgHlYx0Dby+M3WN8kO0GXgtXRZJHAdEDmT/pAEE1j3VNbbvmQpk0NnlclwTwnTyLsfkzZp7FUiRUmG5xAPwDIh3qNgYJeYSAFIsSINSQMifMU3eXgyBM46fuQhRYqLFxQQ3wQVMyBF83BSXbwOnHeb4SNBv7MDt9Xhxmt7fj9tIJDsoEZmbOEhkUgZJEKbFg8Qj8PMNEsTo5UojVwB9tce4DG6rXgbE+i2A/RuBwyvEQFTwCPQH/Y8xFYMAjoHLKWIx5eT9daHf4LQ6+QzMNBaDCWUPtA8eC3PouDUPwsOhTYkkTEJhAD7n6b4vStN0DZ4wt2Hv2a/MpjNtJZUhEwfvZM1kSKguBI01bI0VLKCoh0GnmBwUfF6DY+jtbKcnLvM4cQvM8BX4BdgX4joZpsthXwBGziaP6jecMV9DJSWP+D6ZdiOlQnALyLNDHM5XLW4MoUGlrSuouROmyAAXiyg/pI+dwSIiTl1cr6k0ZSc0wGkdDr+P3xRImVfTSW4FXc2+udpcqN1A5OgwhAlCHfhFVpSqDH1t9lOePBTbKipEVFu/WswBojltBaaQ+b06DwSQTaJNF4jM0DGQmjr0hb7/JjsAhh/ZSigRn5j4DL6eapxwEFXJVvmxO0is9++nZFKTi9mcDfeBtb2pSnZnNW11AcyrMK4040Aacz8MwkIPC0PGJPcz7ikMF12nL1TrpLvbrup3MhdxuPAhMp2lucrjLE8jUe7nofk6IHyszBd0DHDWgNrTGQTsiGEwz9KUWL+Cw+YA0LusG2wBZUhFislaKQr0I9W+F1NNjZgVUfPQ3wMMcJrZzuoAwQjstTVNC1nBKwgPjaGi/Ympk9I45t6zUEJe5N4pR+B5IGHWBzsLnQ8bUH6cDEPWtFGLYV7O6XHT0PnZXPSGUxfnAWwu0NE6k4Oj+hZT6H6RrDLRNJ34/LYAGwb5UdcKvjUO/Ujn0mjp70tX5PE63EuMtrNznepLIGHzxXTK9i53wCypqQzV3U7oQsmnkEzG/jy2AL8guegBsw4OiQnf04CQCkpCIJnoGVLdqwFUDv1lSplUA2tgiIqitLlwqN0dwKh9u8MTJWKluWJ+uThdQ9JfOVh5B2rbC3HY+x1PY6vjtpPzFhIiCQicsd3vsKZQzzkplkTgAht0Fh5uNGk/V3Fm8jwPglIU+0lenK7r5d9ZWRv2JXKia+iwqijZz6GyvcpqJMCAlYPPGBs3n9pqXDSL3DZXiZ7mlTIUCE2xQw3UmRmrxeorpUUw0laW585e9j4JUDacgYUZAhZX2Tu51RiMpwsujicqNuHmKK7yg1QUyld55kdBnBYPkO4mw/k0/k3Wtes2ZT5Ivnmxc6rMbgV5ANmjGltFWQp9i+gbOwzutBiukE1VOgic9zwbWy+ccnT73Ze53FiZcuDKHW6IxQxYt9WvwxE4jv0cazEaXIr2gwftmA3S+jcJsaZvyd0Ff+AIjPojIW9G4O0hNH+CTyH+oDbWaCpncbVEP32+twEXkMVVXV06VspXuMrAdQU73MijPFtML/GzlljEAB4I33+Pa0YHrM6u8o8J5xe4Ze/gWOrm4eKrLQ132jSiMlzGCcaE3biASNXMM4r+uWnQ/UO6QNQj3wkdwpZz5XeRKrDkyGXPfTMSnyzxyhA7kHS48OqGxB0EoBtxHm4/CiP80KiyCOJYmXBfnOGm+zhiHYdQfccYs3H+DYXzq8HyQmFkTcmm9H6VHQlC9i8shynwnWdggsENkHL9MmFj2gjX/VK72m+GzY6amUwDkx8n/z17HfbgFOUn8M2z4w+L6ZuZiyQFwumUpjyP4My3tfm3bO1f5aW9s5V+fhOKMA2Nn2WF75cyelTSXAfNb5uiuBEeFAKyfaO4hY9FVlZlLCL2WJ5/89EBRJoRE6GkeQtNAg5ul7FnmEHD9itehBue99SQ3PGmYXOUJaExSSoCWWaBCnW4X2xbI/cfKQBhl4q0v+NMM0wEYIkbW4CIU0awJ0hqHPhlRPtrY7lsa9wWC4C0mEYKQdvNpX134g4kzrOD1JokD9PUBJk0RhTygZuM1+S6zEJ2SmrjfUcxQGI7Dnu7AzZBsbgBigA+RrMmuNHpDcAAdLyVvJinrSU+jWImQKehQ1zstOnZHyTdL6NYClMWqowyqR68/llT/o+UwWB5DeHUq+5drpgmU3X15AovHdUB5f0pDwKSeK7BTtU12muRqVIbdNl9c2TOkze0BgeJ1TrSChCXn+WFCYp9jCA2b52iAHvalUaITC9ROqvd2wsuLfVzlllggWBoWpb2vOq5Gai/O0//7HpvVIiB/wXZfTIJfbn32gzWV7nPAx9essdcXGNyn84U6+a6sZDwIOPyLPTB34hEyRCiotxnXBBsej4SYkQLue+eG5xr21SfBR8hn2djoH5e8mbFQ+fn4hCpB+yzCQNHlcNP00hCbhSXkcweJMZpvEMiAlvMiDnJ6yiTXSGcAlAlv3DCf3HQQNjTElLkVhcWqiI/BvV+xpzkR8RJMLekE5IkojwDDQHgs+3PLGjad2bLvCO6CEJeUk+vLjaQIxQHuVLmvk7KWAU+QMgyyB+8k/B3YiOGtyD938NfsI5/QNpgSSuQR+H/QQXWb5hmKL1JJg4SHaRGisTPUw2gQIUPqubJm17991i2pxYI4GF86yc70Ioe30+4LmTE2T7RK+ZaBlYPwauw9fLqKYTLCzctF5Tf15Dm4IYegtivCwCRr4s8CsusCPwoCuQDAa8bjR+1vBA9kOTIFq4DFP0EftkaF8KmwbXz4lGWldvi1YE8Xv11lhtGqsy0KUUQ6yQo0gfiWpvuD67QczWckvSG2d+AwYEFugenkJpAQsJPGzH9oIbDOCzKSm4GhrVeYjcJVHKsO9vet0Lks/i6+ubMc1kO2p1HuYY7cM4z0pC9AmZofI2NmSdHtSBOTSp0lId5rLM028d7wl8XIn7CWh5pQeCqLy1qpHEw7dOANUFS9fvMdJpzWmo0ryK94FV7cO6oOhRa52mQ+yYFYFM8wK7Bpuz8EZ8Q6bEfOosVcLgQ4FEyMqpYhpChW3zKRdmgrcfBkI++IPZLpo/8RGqp9TSzn0sf077u/eaGwKOQLSnCXKRSR4jSH8gKMsQVfX+WY0VABJ8HfEGNYYzSyRuWlymxxsejQMeZ9PG76dnsjLDxvaI3lTtiMa+d8nmXKPJrHXp2BGYRh/E0cXLy1BlB/x7l903WWHcaxVmFQVRE7ZOO6gPA5Os2oMVZetRZFKUSaRJkZZJHIhQPtGPj3G1Z4r1QjlN3CZ+YS8e6zuBjY3qk/UuN1QF859g7JBMStnGhyr10rs0XKl3SJHYr6dQLuM8lzxFF81jRDMPara5mY9ifC0mymjpIv/0udJdRaRxERSBDCIUy1fEDsBxEGzVTDQf2fK6lUa8M+RDPQyjtJXBxww638IMJB20NHFi/sZX4aCgHCKRSP5BZFqR+Ee2junCN+P9iCqOUvDct+rayaDmLxuL01JjQa0gd/RYEMy3lWH+RHRWg+c07LVhQfX+qTHGcttjxe0ddeCEilaQ+QMUkiAuZPHALa6SWUMP9xWXnaP7avbfvLbd6pmI7vto6zQ0rmke9LSAfc3dqfmzjvdubk3ZaIh/svNHRw1ErLA3854tAxeYBZA/XQevTKjU7mtCJTfZFKV39OKNbJo13S9zP637jMnuZ3SbMH63qFJDBCWz8lVke54H/wPdtsB7k6jX+VQE9KosXzUp8AEUfUXkoxE+z5r8YSe4dTPf2haDwwn8t+ptuC1eknCZ/nWqUIonToIyyoCyK8hGpfrRphNXkjcjKe+duQc3dG8prBwDWdP0CCybJPEpTwny9apjJOHn/gk9OHQxK3myeZPed3Kupu7tZaMs8CTmm6cLs7wcukn4dpihykQnfRKUwYWby/T8CEFH6/wmhDaDk5A5/3S5vkLHrhN+R9/R3BdMTu/t23LSclTGIdpp1h3NCQDJhFIcqixJfJfuJsAj97WOGG1bACvnCf76GHaTpb50U4nvuM+NFAUQrbHEiNBufAWqhVuTzWFk1Yzc9dOX+O5ZWQFUCsZ40v/dd7vamjrQ6au5BYe/ZvU38OrtTYaaCKMjDOAqjQO2nzpvnZjEwQ89ZAN0+gHCIuG8evHcaCtm/mOFNeXg1nSLOatPt+jV1CVD3H38Hawu8ig==')
assert len(CASES)==48 and {c['dataset'] for c in CASES}=={'gsm8k','svamp'}
assert len({c['id'] for c in CASES})==48
assert sha(MANIFEST)=='659c32eefa5540c959ffe21a639e238b7998e5e21b63870831b60016cb61cdaf'
assert all(c in MANIFEST['cases'] for c in CASES)
BANK={c['id']:c for c in CASES}

def atomic_json(path,obj):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        assert json.loads(path.read_text())==obj,'Immutable checkpoint collision: '+str(path)
    else:
        tmp=path.with_name(path.name+'.writing')
        tmp.write_text(json.dumps(obj,sort_keys=True,indent=2,ensure_ascii=False))
        os.replace(tmp,path)

def get_record(tid):
    path=RECS/(hashlib.sha256(tid.encode()).hexdigest()+'.json')
    if not path.exists():return None
    obj=json.loads(path.read_text());assert obj['trial_id']==tid and obj['protocol_hash']==PROTOCOL_HASH
    return obj

def save_record(obj):
    assert obj['protocol_hash']==PROTOCOL_HASH
    atomic_json(RECS/(hashlib.sha256(obj['trial_id'].encode()).hexdigest()+'.json'),obj)

def all_records():
    rows=[json.loads(x.read_text()) for x in RECS.glob('*.json')]
    assert len(rows)==len({x['trial_id'] for x in rows})
    assert all(x['protocol_hash']==PROTOCOL_HASH for x in rows)
    return rows

def grade_explicit(output,target):
    t=str(output)
    patterns=[r'(?im)^\s*FINAL_ANSWER\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$',
              r'\\boxed\s*\{\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*\}',
              r'(?im)^\s*(?:final )?answer\s*:\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+))\s*[.!]?\s*$']
    nums=[]
    for pattern in patterns:
        nums += [float(x) for x in re.findall(pattern,t)]
    if not nums or any(not math.isclose(nums[0],x,rel_tol=1e-9,abs_tol=1e-9) for x in nums):
        return {'label':'ambiguous','value':None}
    return {'label':'correct' if math.isclose(nums[0],float(target),abs_tol=1e-8,rel_tol=1e-8) else 'wrong','value':nums[0]}

def export_zip():
    zpath=Path('/content')/(STUDY+'_Snapshot.zip')
    with zipfile.ZipFile(zpath,'w',zipfile.ZIP_DEFLATED) as z:
        for p in ROOT.rglob('*'):
            if p.is_file() and not p.name.endswith('.writing'):z.write(p,p.relative_to(ROOT))
    print('Saved snapshot',zpath,zpath.stat().st_size,'bytes')
    try:
        from google.colab import files
        files.download(str(zpath))
    except Exception as ex:print('Manual download available:',repr(ex))

# Corrected scorer has separate strict complete and semantic human adjudication layers.
assert grade_explicit('FINAL_ANSWER: 17',17)['label']=='correct'
assert grade_explicit('FINAL_ANSWER: 18',17)['label']=='wrong'
assert grade_explicit('Working: 17',17)['label']=='ambiguous'


In [ ]:
from types import SimpleNamespace
from huggingface_hub import snapshot_download
REPO_DIR=Path('/content/ALoDLM_Q1_Final')
CODE_COMMIT='fe225a99e0b5aeff351804e871d7581155d7f27d'
if not (REPO_DIR/'.git').exists():
    subprocess.run(['git','clone','-q','https://github.com/amazon-science/ALoDLM.git',str(REPO_DIR)],check=True)
subprocess.run(['git','-C',str(REPO_DIR),'checkout','--detach','-q',CODE_COMMIT],check=True)
assert subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()==CODE_COMMIT
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'--no-deps'],check=True)
sys.path.insert(0,str(REPO_DIR))
from alodlm.generate import load_decoder
from alodlm.inference import DecodeConfig
MODEL_ID='amazon/ALoDLM-1.7B';MODEL_REV='588c7dc1946e3e09fd496da0b0808bd26f46a758'
MODEL_PATH=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REV)
args=SimpleNamespace(model=MODEL_PATH,model_config=None,device='cuda',precision='fp32')
decoder=load_decoder(args);tokenizer=decoder.tokenizer
assert next(decoder.model.parameters()).dtype==torch.float32

def model_generate(prompt,budget,seed):
    random.seed(seed);np.random.seed(seed%(2**32-1));torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
    ids=list(tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,enable_thinking=False))
    config=DecodeConfig(q=0.4,tau=0.4,mode='left1',window_size=16,max_new_tokens=budget,position_penalty=None)
    config.validate()
    stamp=time.monotonic()
    result=decoder.generate(ids,config)
    text=tokenizer.decode(list(map(int,result['token_ids'])),skip_special_tokens=True).strip()
    return {'output':text,'nfe':int(result.get('outer_steps',0)),
        'loop_token_passes':int(result.get('loop_token_passes',0)),
        'generated_tokens':int(result.get('generated_tokens',0)),
        'stop_reason':str(result.get('stop_reason','unknown')),
        'wall_seconds':round(time.monotonic()-stamp,3)}


In [ ]:
STUDY='RSX_R06_ALoDLM_Budget_Control'
ROOT=Path('/content/drive/MyDrive/RepairScopeX/R06_Equal_Budget_Control')
RECS=ROOT/'records';RECS.mkdir(parents=True,exist_ok=True)
PHASES=('source128','source256','finalize128','regenerate128')
BUDGETS={'source128':128,'source256':256,'finalize128':128,'regenerate128':128}
COMPARISONS=[('source128','regenerate128'),('source128','finalize128'),('source256','finalize128')]
SETTINGS={'model_revision':MODEL_REV,'code_commit':CODE_COMMIT,'decoder':'left1','q':0.4,'tau':0.4,
  'window':16,'precision':'fp32','budgets':BUDGETS,
  'scope':'48 GSM8K/SVAMP cases frozen from R02 manifest',
  'warning':'128+128 separate inference is not compute-identical to 256-token single output; report measured passes'}
PLAN=[(c['id'],ph) for c in CASES for ph in PHASES]
PROTOCOL={'study':STUDY,'selection_sha256':sha(CASES),'manifest_sha256':sha(MANIFEST),'plan':PLAN,'settings':SETTINGS}
PROTOCOL_HASH=sha(PROTOCOL)
atomic_json(ROOT/'protocol.json',{'protocol':PROTOCOL,'protocol_hash':PROTOCOL_HASH})
atomic_json(ROOT/'cases.json',CASES)
print('Frozen protocol',PROTOCOL_HASH,'existing',len(all_records()),'/',len(PLAN))


In [ ]:
def prompt_for(case,phase):
    question=case['prompt']
    basic='\nGive a brief solution and finish with FINAL_ANSWER: <number>.'
    if phase in ('source128','source256'):return question+basic
    if phase=='regenerate128':return question+'\nSolve independently from scratch; check the numerical result. End with FINAL_ANSWER: <number>.'
    if phase=='finalize128':
        initial=get_record(STUDY+'|'+case['id']+'|source128')
        if initial is None:raise RuntimeError('Missing source128 for finalization: '+case['id'])
        return question+'\nA previous response was:\n'+str(initial['output'])[:1300]+'\nCheck this response and give the correct final number as FINAL_ANSWER: <number>.'
    raise ValueError(phase)


In [ ]:
SESSION_MAX_NEW=12
SESSION_MAX_MINUTES=27
start=time.monotonic();new=0;err=None
for problem_id,phase in PLAN:
    if new>=SESSION_MAX_NEW or time.monotonic()-start>=SESSION_MAX_MINUTES*60:break
    trial=STUDY+'|'+problem_id+'|'+phase
    if get_record(trial) is not None:continue
    case=BANK[problem_id]
    try:
        prompt=prompt_for(case,phase)
        seed=20261025+int(hashlib.sha256(problem_id.encode()).hexdigest()[:8],16)%1000000
        result=model_generate(prompt,BUDGETS[phase],seed)
        record={'trial_id':trial,'problem_id':problem_id,'phase':phase,'dataset':case['dataset'],
             'protocol_hash':PROTOCOL_HASH,'seed':seed,'prompt_sha256':hashlib.sha256(prompt.encode()).hexdigest(),
             'grade':grade_explicit(result['output'],case['target']), 'target':case['target'],**result}
        save_record(record);new+=1
        print('SAVED',new,problem_id,phase,record['grade']['label'],flush=True)
    except Exception as exc:
        err={'trial_id':trial,'error':repr(exc)}
        (ROOT/'last_error.json').write_text(json.dumps(err,indent=2))
        if 'out of memory' in str(exc).lower():torch.cuda.empty_cache()
        print('STOP after saved records. Error:',repr(exc));break
print('New calls this session',new,'total',len(all_records()),'/',len(PLAN))


In [ ]:
from scipy.stats import binomtest
rec=all_records();lookup={(x['problem_id'],x['phase']):x for x in rec}
complete=[c for c in CASES if all((c['id'],ph) in lookup for ph in PHASES)]
rows=[]
for case in complete:
    row={'problem_id':case['id'],'dataset':case['dataset']}
    for ph in PHASES:
        r=lookup[(case['id'],ph)]
        row[ph+'_correct']=int(r['grade']['label']=='correct')
        row[ph+'_ambiguous']=int(r['grade']['label']=='ambiguous')
        row[ph+'_nfe']=r['nfe'];row[ph+'_loop_token_passes']=r['loop_token_passes'];row[ph+'_time']=r['wall_seconds']
        row[ph+'_total_passes']=r['loop_token_passes'] + (lookup[(case['id'],'source128')]['loop_token_passes'] if ph=='finalize128' else 0)
        row[ph+'_total_nfe']=r['nfe'] + (lookup[(case['id'],'source128')]['nfe'] if ph=='finalize128' else 0)
    rows.append(row)
df=pd.DataFrame(rows)
if not df.empty:
    df.to_csv(ROOT/'matched_results.csv',index=False)
    print('Complete paired problems',len(df),'/',len(CASES))
    print(df.groupby('dataset')[[ph+'_correct' for ph in PHASES]].agg(['count','sum']))
else:print('No completed matched problem sets yet')
if len(complete)==len(CASES):
    from scipy.stats import binomtest
    stats={}
    for x,y in COMPARISONS:
        a=df[x+'_correct'].to_numpy();b=df[y+'_correct'].to_numpy()
        wins=int(((b==1)&(a==0)).sum());losses=int(((b==0)&(a==1)).sum())
        stats[f'{y}_vs_{x}']={'n':len(a),'reference_correct':int(a.sum()),'comparison_correct':int(b.sum()),
          'recoveries':wins,'damage':losses,'paired_exact_p':float(binomtest(min(wins,losses),wins+losses,.5).pvalue) if wins+losses else 1.0,
          'mean_total_passes_ref':float(df[x+'_total_passes'].mean()),
          'mean_total_passes_comparison':float(df[y+'_total_passes'].mean()),
          'mean_total_nfe_reference':float(df[x+'_total_nfe'].mean()),
          'mean_total_nfe_comparison':float(df[y+'_total_nfe'].mean())}
    (ROOT/'FINAL_COMPARISONS.json').write_text(json.dumps(stats,indent=2))
    print(json.dumps(stats,indent=2))
export_zip()
